# PathNet Tutorial: Multi-Scale CNN for Pathology Patch Classification

**Author:** Hamed Abdollahi  
**Environment:** Python 3.9 · TensorFlow 2.12

---

## What this notebook teaches

This is a self-contained, runnable tutorial for building a patch-level classifier
for H&E histopathology images. No real slide data is needed — we simulate
realistic patches so you can run everything immediately.

By the end you will understand:

| Section | Concept |
|---|---|
| 1 | Why H&E patch classification is hard (and different from natural images) |
| 2 | Simulating realistic pathology patches with class imbalance |
| 3 | Stain normalisation — what it is and why it matters |
| 4 | Squeeze-and-Excitation: channel recalibration visualised |
| 5 | Multi-scale fusion: why one kernel size is not enough |
| 6 | Spatial Attention Pooling: seeing where the model looks |
| 7 | Learned Positional Encoding: giving the CNN a spatial memory |
| 8 | Class imbalance: Focal Loss + class weights + oversampling |
| 9 | Two-stage fine-tuning: warm-up then unfreeze |
| 10 | Evaluation: AUC, optimal threshold, confusion matrix, attention maps |

---

> **Design philosophy:** Each section first explains the *biological problem*,
> then shows the *engineering solution*, then *visualises the effect*.
> Code comments explain *why* each line exists, not just what it does.

## 0. Setup

Install and import everything needed. All dependencies are standard — no
specialist pathology libraries required.

In [ ]:
# Install any missing packages
import subprocess, sys
required = ["tensorflow", "numpy", "matplotlib", "scikit-learn", "opencv-python", "scipy"]
for pkg in required:
    try:
        __import__(pkg.replace("-", "_"))
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

import numpy as np
import cv2
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import tensorflow as tf
from tensorflow import keras
import tensorflow.keras.layers as tfkl
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score, roc_curve, f1_score,
    confusion_matrix, ConfusionMatrixDisplay,
    classification_report, precision_recall_curve,
)

# Reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print(f"TensorFlow version : {tf.__version__}")
print(f"GPUs available     : {len(tf.config.list_physical_devices('GPU'))}")

---
## 1. Why H&E Patch Classification is Hard

H&E (Haematoxylin and Eosin) is the most common stain in pathology.
- **Haematoxylin** stains nuclei blue/purple
- **Eosin** stains cytoplasm and extracellular matrix pink

Three properties make it harder than classifying cats vs dogs:

| Problem | Why it matters |
|---|---|
| **Multi-scale features** | Cancer is visible at the nucleus level (~10 µm) AND the tissue architecture level (~500 µm) simultaneously. A single CNN kernel cannot capture both. |
| **Stain variation** | The same tissue stained in different labs or scanned on different scanners looks different in colour — the model must not learn scanner-specific biases. |
| **Class imbalance** | In most whole-slide images, cancerous regions cover 10–30% of tissue. A naive model learns to always predict "benign" and still achieves 80%+ accuracy. |

Every architectural choice in PathNet addresses one of these three problems.
Let's start by creating data that captures all three.

---
## 2. Simulating Realistic H&E Patches

We generate synthetic patches that mimic the key visual properties of H&E slides:
- **Background**: pale pink (eosin-like)
- **Nuclei**: small dark blue/purple circular blobs (haematoxylin-like)
- **Cancerous patches**: denser, more irregular nuclei; darker staining
- **Class imbalance**: 25% cancerous, 75% benign — realistic for a WSI
- **Stain variation**: random colour shift per patch to simulate different scanners

In [ ]:
def simulate_he_patch(patch_size=64, is_cancerous=False, scanner_shift=0.0):
    """
    Generate a synthetic H&E-like patch.

    Benign patches have ~8 sparse, round nuclei.
    Cancerous patches have ~20 dense, irregular nuclei with darker staining.
    scanner_shift adds a colour bias to simulate different scanners.
    """
    # Start with a pale pink background (eosin-like)
    patch = np.ones((patch_size, patch_size, 3), dtype=np.float32)
    patch[:, :, 0] = 0.85 + np.random.normal(0, 0.03, (patch_size, patch_size))  # R
    patch[:, :, 1] = 0.65 + np.random.normal(0, 0.03, (patch_size, patch_size))  # G
    patch[:, :, 2] = 0.75 + np.random.normal(0, 0.03, (patch_size, patch_size))  # B

    # Add nuclei (dark blue/purple blobs)
    n_nuclei  = np.random.randint(15, 25) if is_cancerous else np.random.randint(5, 10)
    for _ in range(n_nuclei):
        cx = np.random.randint(5, patch_size - 5)
        cy = np.random.randint(5, patch_size - 5)
        # Cancerous nuclei are larger and more irregular
        radius = np.random.randint(3, 7) if is_cancerous else np.random.randint(2, 4)
        for y in range(max(0, cy - radius), min(patch_size, cy + radius)):
            for x in range(max(0, cx - radius), min(patch_size, cx + radius)):
                # Add irregularity for cancerous nuclei
                noise = np.random.uniform(0.7, 1.0) if is_cancerous else 1.0
                dist  = np.sqrt((x - cx)**2 + (y - cy)**2)
                if dist < radius * noise:
                    darkness = 0.15 if is_cancerous else 0.25
                    patch[y, x, 0] = darkness + np.random.normal(0, 0.02)   # dark R
                    patch[y, x, 1] = darkness + np.random.normal(0, 0.02)   # dark G
                    patch[y, x, 2] = 0.45 + np.random.normal(0, 0.03)       # blue B

    # Scanner colour shift — simulates staining variation between labs
    patch[:, :, 0] += scanner_shift
    patch[:, :, 2] -= scanner_shift * 0.5

    return np.clip(patch, 0, 1)


def generate_dataset(n_patches=500, patch_size=64, pos_fraction=0.25,
                     n_scanners=3):
    """
    Generate a dataset with:
      - n_patches total patches
      - pos_fraction cancerous (minority class)
      - n_scanners different colour shifts (simulating lab variation)
    """
    X, y = [], []
    # Each scanner has a fixed colour shift
    scanner_shifts = np.linspace(-0.08, 0.08, n_scanners)

    for i in range(n_patches):
        is_cancerous  = (np.random.rand() < pos_fraction)
        scanner_shift = scanner_shifts[i % n_scanners]
        patch = simulate_he_patch(patch_size, is_cancerous, scanner_shift)
        X.append(patch)
        y.append(1.0 if is_cancerous else 0.0)

    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)


# Generate dataset: 500 patches, 25% cancerous, 3 scanner types
PATCH_SIZE = 64
X, y = generate_dataset(n_patches=500, patch_size=PATCH_SIZE,
                         pos_fraction=0.25, n_scanners=3)

print(f"Dataset shape : {X.shape}")
print(f"Labels        : {int(np.sum(y==0))} benign  /  {int(np.sum(y==1))} cancerous")
print(f"Imbalance     : {np.sum(y==1)/len(y)*100:.1f}% positive")

In [ ]:
# Visualise example patches from each class and scanner
fig, axes = plt.subplots(2, 6, figsize=(14, 5))
fig.suptitle("Simulated H&E patches\n"
             "Top: Benign (sparse, round nuclei)   "
             "Bottom: Cancerous (dense, irregular nuclei)",
             fontsize=12, fontweight="bold")

benign_idx     = np.where(y == 0)[0][:6]
cancerous_idx  = np.where(y == 1)[0][:6]

for col, (bi, ci) in enumerate(zip(benign_idx, cancerous_idx)):
    axes[0, col].imshow(X[bi])
    axes[0, col].set_title(f"Benign\n(scanner {col % 3 + 1})", fontsize=9)
    axes[0, col].axis("off")
    axes[1, col].imshow(X[ci])
    axes[1, col].set_title(f"Cancerous\n(scanner {col % 3 + 1})", fontsize=9,
                            color="red")
    axes[1, col].axis("off")

plt.tight_layout()
plt.show()

print("\nNotice: the same class looks different across scanners (colour shift).")
print("Stain normalisation corrects this before training.")

---
## 3. Stain Normalisation (Macenko SVD Method)

### The problem
A model trained on scanner 1's colour profile will fail on scanner 2 — not because
the tissue is different, but because the *colour* is different. This is called
**domain shift** and it is the most common silent failure mode in pathology ML.

### The solution: Macenko 2009
Instead of working in RGB space, Macenko works in **Optical Density (OD)** space:
```
OD = -log(I / 255)
```
In OD space, H (haematoxylin) and E (eosin) stains are approximately orthogonal
vectors. SVD finds these two directions. We then decompose each patch into
H-concentration and E-concentration, and reconstruct using a fixed reference's
stain vectors — making all patches look like they came from the same lab.

In [ ]:
class MacenkoNormaliser:
    """
    SVD-based H&E stain normalisation (Macenko et al., 2009).

    Steps:
      1. Convert RGB → Optical Density (OD)
      2. Remove near-transparent (background) pixels
      3. SVD → find the 2D stain plane
      4. Extract H and E direction vectors from percentile angles
      5. Decompose patch into stain concentrations
      6. Reconstruct with reference stain vectors
    """

    def __init__(self, beta=0.15, alpha=1.0):
        self.beta  = beta    # OD threshold for background removal
        self.alpha = alpha   # percentile for robust angle estimation
        self._HERef   = None
        self._maxCRef = None

    @staticmethod
    def _to_od(patch):
        """Float32 [0,1] patch → optical density matrix (N_pixels × 3)."""
        p  = np.maximum(patch.reshape(-1, 3).astype(np.float64), 1e-6)
        return -np.log(p)

    @staticmethod
    def _stain_matrix(od, beta, alpha):
        """Extract H and E stain direction vectors via SVD."""
        od_hat = od[np.all(od > beta, axis=1)]  # keep tissue pixels only
        if len(od_hat) < 10:
            return np.array([[0.650, 0.704, 0.286],
                             [0.268, 0.570, 0.776]])
        _, _, V = np.linalg.svd(od_hat, full_matrices=False)
        V       = V[:2]                          # top-2 singular vectors
        that    = od_hat @ V.T                   # project onto stain plane
        phi     = np.arctan2(that[:, 1], that[:, 0])
        v1 = V.T @ [np.cos(np.percentile(phi, alpha)),
                    np.sin(np.percentile(phi, alpha))]
        v2 = V.T @ [np.cos(np.percentile(phi, 100 - alpha)),
                    np.sin(np.percentile(phi, 100 - alpha))]
        # H vector has higher red-channel OD value
        return np.array([v1, v2]) if v1[0] > v2[0] else np.array([v2, v1])

    def fit(self, reference_patch):
        """Fit to a representative reference patch (float32, [0,1])."""
        od = self._to_od(reference_patch)
        self._HERef  = self._stain_matrix(od, self.beta, self.alpha)
        od_hat       = od[np.all(od > self.beta, axis=1)]
        C            = np.linalg.lstsq(self._HERef.T, od_hat.T, rcond=None)[0]
        self._maxCRef = np.percentile(C, 99, axis=1)
        return self

    def transform(self, patch):
        """Normalise a patch to the reference appearance (float32 [0,1] → float32 [0,1])."""
        if self._HERef is None:
            raise RuntimeError("Call .fit() first.")
        H, W   = patch.shape[:2]
        od     = self._to_od(patch)
        HE     = self._stain_matrix(od, self.beta, self.alpha)
        C      = np.linalg.lstsq(HE.T, od.T, rcond=None)[0]
        maxC   = np.percentile(C, 99, axis=1, keepdims=True) + 1e-8
        C      = C / maxC * self._maxCRef[:, None]
        od_n   = self._HERef.T @ C
        rgb_n  = np.clip(np.exp(-od_n.T).reshape(H, W, 3), 0, 1)
        return rgb_n.astype(np.float32)


# Fit normaliser to a "reference" benign patch (scanner 0 — no shift)
ref_idx    = np.where(y == 0)[0][0]
normaliser = MacenkoNormaliser().fit(X[ref_idx])
print("MacenkoNormaliser fitted to reference patch.")
print(f"  Reference H stain vector: {normaliser._HERef[0].round(3)}")
print(f"  Reference E stain vector: {normaliser._HERef[1].round(3)}")

In [ ]:
# Visualise: same tissue, 3 scanners, before and after normalisation
# Pick one cancerous patch from each scanner
cancerous_by_scanner = [np.where(y == 1)[0][i * 5] for i in range(3)]

fig, axes = plt.subplots(2, 3, figsize=(10, 6))
fig.suptitle("Stain Normalisation Effect\n"
             "Top: raw patches from 3 scanners   "
             "Bottom: after Macenko normalisation",
             fontsize=12, fontweight="bold")

for col, idx in enumerate(cancerous_by_scanner):
    raw  = X[idx]
    norm = normaliser.transform(raw)

    axes[0, col].imshow(raw)
    axes[0, col].set_title(f"Scanner {col + 1} — raw", fontsize=10)
    axes[0, col].axis("off")

    axes[1, col].imshow(norm)
    axes[1, col].set_title(f"Scanner {col + 1} — normalised", fontsize=10)
    axes[1, col].axis("off")

plt.tight_layout()
plt.show()

# Apply normalisation to full dataset
X_norm = np.array([normaliser.transform(x) for x in X], dtype=np.float32)
print(f"\nNormalised dataset shape: {X_norm.shape}")
print("After normalisation, all patches share the same colour reference.")

---
## 4. Building Block 1: Squeeze-and-Excitation

### The problem
A standard Conv layer treats all feature channels equally. But in pathology,
some channels encode useful information (nuclear texture, chromatin pattern)
while others encode noise (staining artefacts, scanner variation).

### The solution: SE block (Hu et al., 2018)
```
Input (B, H, W, C)
  → Global Average Pool → (B, C)          ← "Squeeze": describe each channel globally
  → Dense(C//ratio) → ReLU → Dense(C)     ← "Excite": learn channel importance
  → Sigmoid → reshape (B, 1, 1, C)
  → Multiply with input                   ← "Scale": amplify useful, suppress noise
```

The key insight: a channel that is uniformly activated across the entire spatial
map is probably noise (staining variation); a channel that activates strongly
in specific locations is probably encoding a real feature.

In [ ]:
class SqueezeExcitationBlock(tfkl.Layer):
    """
    Channel-wise feature recalibration.

    ratio=16 is the standard from the SE-Net paper. Higher ratio → fewer
    parameters but coarser recalibration. For pathology, ratio=8 or 16 works well.
    """

    def __init__(self, ratio=16, **kwargs):
        super().__init__(**kwargs)
        self.ratio = ratio

    def build(self, input_shape):
        C = input_shape[-1]
        self.gap     = tfkl.GlobalAveragePooling2D()
        self.fc1     = tfkl.Dense(max(1, C // self.ratio), activation="relu",  use_bias=False)
        self.fc2     = tfkl.Dense(C,                       activation="sigmoid", use_bias=False)
        self.reshape = tfkl.Reshape((1, 1, C))

    def call(self, inputs):
        x = self.gap(inputs)      # (B, C)      — global channel descriptor
        x = self.fc1(x)           # (B, C//r)   — bottleneck
        x = self.fc2(x)           # (B, C)      — per-channel gate in (0,1)
        x = self.reshape(x)       # (B,1,1,C)   — broadcast over spatial dims
        return inputs * x         # (B, H, W, C) — rescaled feature map

    def get_config(self):
        return {**super().get_config(), "ratio": self.ratio}


# Visualise: what does the SE block actually do to channels?
# Build a tiny model: random conv features → SE → show gate values
inp    = keras.Input((PATCH_SIZE, PATCH_SIZE, 16))
se_out = SqueezeExcitationBlock(ratio=4)(inp)
se_model = keras.Model(inp, se_out)

# Create dummy feature map: 16 channels, some noisy (uniform), some informative (localised)
dummy = np.zeros((1, PATCH_SIZE, PATCH_SIZE, 16), dtype=np.float32)
for c in range(16):
    if c < 8:
        # Noisy channels: uniform activation across spatial dim
        dummy[0, :, :, c] = np.random.uniform(0.4, 0.6, (PATCH_SIZE, PATCH_SIZE))
    else:
        # Informative channels: localised activation (simulating nuclear response)
        dummy[0, 20:40, 20:40, c] = np.random.uniform(0.8, 1.0, (20, 20))

output = se_model.predict(dummy, verbose=0)

# The SE gate values — channels with localised activations should get higher gates
# (The actual gate values depend on random initialisation at this point,
# but the architecture ensures learned gates after training)

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
fig.suptitle("Squeeze-and-Excitation Block — effect on feature channels",
             fontsize=12, fontweight="bold")

axes[0].imshow(dummy[0, :, :, 0], cmap="hot")   # noisy channel
axes[0].set_title("Input: noisy channel (uniform)\n→ low SE gate")
axes[0].axis("off")

axes[1].imshow(dummy[0, :, :, 8], cmap="hot")   # informative channel
axes[1].set_title("Input: informative channel (localised)\n→ high SE gate")
axes[1].axis("off")

axes[2].imshow(output[0, :, :, 8] - dummy[0, :, :, 8], cmap="RdBu_r")
axes[2].set_title("Output difference\n(SE amplifies informative regions)")
axes[2].axis("off")

plt.tight_layout()
plt.show()

print("After training, SE gates are LOW for staining-noise channels")
print("and HIGH for channels encoding nuclear morphology or texture.")

---
## 5. Building Block 2: Multi-Scale Fusion

### The problem
A 3×3 kernel sees a ~15 µm region at 20× magnification — enough to characterise
a single nucleus. But gland architecture (disrupted in cancer) spans ~200 µm.
A 7×7 kernel sees ~35 µm. **No single kernel size is sufficient.**

### The solution: parallel branches + concatenation
Run three kernel sizes in parallel, concatenate the results, then project
back to the original channel count with a 1×1 Conv. The network learns
which scale matters most for each output channel.

**Why concatenate and not sum?**  
Sum fusion assumes the three scales encode the same information at different
resolutions. Concat assumes they may encode complementary information — and
lets the 1×1 Conv learn the combination. For pathology, where nuclear detail
and tissue architecture are qualitatively different, concat is safer.

In [ ]:
class MultiScaleFusionBlock(tfkl.Layer):
    """
    Parallel Conv 3×3 / 5×5 / 7×7, concatenated then projected.

    Each branch uses padding='same' to preserve spatial dimensions (H, W).
    After concatenation we have 3×filters channels; the 1×1 Conv projects
    back to `filters` channels so the output matches the input width.
    A residual connection is added for gradient flow.
    """

    def __init__(self, filters, se_ratio=16, use_bn=True, **kwargs):
        super().__init__(**kwargs)
        self.filters  = filters
        self.se_ratio = se_ratio
        self.use_bn   = use_bn

    def _branch(self, k):
        """Conv k×k → BN → ReLU."""
        layers = [tfkl.Conv2D(self.filters, k, padding="same",
                              use_bias=not self.use_bn)]
        if self.use_bn:
            layers.append(tfkl.BatchNormalization())
        layers.append(tfkl.Activation("relu"))
        return keras.Sequential(layers)

    def build(self, input_shape):
        self.b3 = self._branch(3)
        self.b5 = self._branch(5)
        self.b7 = self._branch(7)

        # 1×1 projection: 3×filters → filters
        self.fuse = keras.Sequential([
            tfkl.Conv2D(self.filters, 1, padding="same",
                        use_bias=not self.use_bn),
            tfkl.BatchNormalization() if self.use_bn else tfkl.Lambda(lambda x: x),
            tfkl.Activation("relu"),
        ])
        self.se = SqueezeExcitationBlock(ratio=self.se_ratio)

        # Residual: align input channels to `filters` if different
        self.proj = (tfkl.Conv2D(self.filters, 1, padding="same", use_bias=False)
                     if input_shape[-1] != self.filters else None)

    def call(self, inputs, training=False):
        f3    = self.b3(inputs, training=training)
        f5    = self.b5(inputs, training=training)
        f7    = self.b7(inputs, training=training)
        fused = tf.concat([f3, f5, f7], axis=-1)    # (B, H, W, 3×filters)
        fused = self.fuse(fused, training=training)  # (B, H, W, filters)
        fused = self.se(fused)
        skip  = self.proj(inputs) if self.proj else inputs
        return tf.nn.relu(fused + skip)

    def get_config(self):
        return {**super().get_config(),
                "filters": self.filters,
                "se_ratio": self.se_ratio,
                "use_bn": self.use_bn}


# Visualise: what each kernel size responds to
# Apply three separate convolutions to a patch and compare activation maps
patch_sample = X_norm[np.where(y == 1)[0][0]]

fig, axes = plt.subplots(1, 4, figsize=(14, 4))
fig.suptitle("Multi-Scale Fusion — why one kernel size is not enough",
             fontsize=12, fontweight="bold")

axes[0].imshow(patch_sample)
axes[0].set_title("Original patch"); axes[0].axis("off")

for col, (k, title) in enumerate([
    (3, "3×3 kernel\n(nuclei detail)"),
    (5, "5×5 kernel\n(cluster level)"),
    (7, "7×7 kernel\n(architecture)"),
]):
    # Simple edge detection to illustrate scale sensitivity
    gray   = cv2.cvtColor((patch_sample * 255).astype(np.uint8), cv2.COLOR_RGB2GRAY)
    kernel = np.ones((k, k), np.float32) / (k * k)
    resp   = cv2.filter2D(gray.astype(np.float32), -1, kernel)
    axes[col + 1].imshow(np.abs(gray.astype(np.float32) - resp), cmap="hot")
    axes[col + 1].set_title(title); axes[col + 1].axis("off")

plt.tight_layout()
plt.show()
print("Fusing all three scales captures both fine nuclear detail and coarse structure.")

---
## 6. Building Block 3: Spatial Attention Pooling

### The problem
Global Average Pooling (GAP) reduces a feature map `(B, H, W, C)` to `(B, C)` by
averaging every spatial location equally. But in a 224×224 patch, a cancerous nucleus
might occupy 10×10 pixels — less than 0.2% of the image. GAP dilutes this signal
with ~99.8% uninformative background.

### The solution: learned spatial attention
Generate a weight map `α(h, w)` — one scalar per spatial location — and use it
to compute a **weighted average** instead of a uniform average.

```
α(h,w) = sigmoid(Conv1×1(F(h,w)))    ← learned, one weight per location
output = Σ α(h,w) · F(h,w) / Σ α(h,w)   ← normalised weighted sum
```

The attention map `α` is interpretable: after training it highlights the
spatial regions that drove the classification decision.

In [ ]:
class SpatialAttentionPooling(tfkl.Layer):
    """
    Attention-weighted pooling replacing plain GlobalAveragePooling2D.

    The attention map is stored in self.last_attention after each forward pass
    so it can be extracted for visualisation without rebuilding the model.
    """

    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.last_attention = None

    def build(self, input_shape):
        # 1×1 conv: one attention weight per spatial location
        self.attn_conv = tfkl.Conv2D(1, kernel_size=1, activation="sigmoid")

    def call(self, inputs, training=False):
        alpha = self.attn_conv(inputs)          # (B, H, W, 1)
        self.last_attention = alpha             # cache for external visualisation
        weighted    = inputs * alpha            # (B, H, W, C)
        sum_weights = tf.reduce_sum(alpha, axis=[1, 2], keepdims=True) + 1e-8
        pooled      = tf.reduce_sum(weighted, axis=[1, 2]) / tf.squeeze(
            sum_weights, axis=[1, 2]
        )                                       # (B, C)
        return pooled


# Visualise: compare GAP vs attention pooling on the same patch
# Build a tiny model that outputs both the attention map and a pooled vector
inp        = keras.Input((PATCH_SIZE, PATCH_SIZE, 8))
attn_layer = SpatialAttentionPooling(name="sap")
pooled_out = attn_layer(inp)
attn_model = keras.Model(inp, pooled_out)

# Create a feature map where one corner has a strong localised signal ("tumour")
feat_map = np.random.uniform(0.1, 0.3, (1, PATCH_SIZE, PATCH_SIZE, 8)).astype(np.float32)
feat_map[0, 10:25, 10:25, :] = np.random.uniform(0.8, 1.0, (15, 15, 8))  # "lesion"

_ = attn_model.predict(feat_map, verbose=0)
attn_map = attn_layer.last_attention.numpy()[0, :, :, 0]

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
fig.suptitle("Spatial Attention Pooling vs Plain GAP",
             fontsize=12, fontweight="bold")

axes[0].imshow(feat_map[0, :, :, 0], cmap="hot")
axes[0].set_title("Feature map\n(bright region = 'lesion')")
axes[0].axis("off")

axes[1].imshow(attn_map, cmap="jet")
axes[1].set_title("Attention weights α(h,w)\n(red = high weight)")
axes[1].axis("off")
plt.colorbar(cm.ScalarMappable(cmap="jet"), ax=axes[1], fraction=0.046)

# Show what GAP would have contributed vs attention pooling
gap_weights = np.ones((PATCH_SIZE, PATCH_SIZE)) / (PATCH_SIZE ** 2)
diff        = attn_map - gap_weights / gap_weights.max() * attn_map.max()
axes[2].imshow(diff, cmap="RdBu_r")
axes[2].set_title("Attention vs GAP weight diff\n(red = SAP upweights here)")
axes[2].axis("off")

plt.tight_layout()
plt.show()
print("SAP concentrates pooling weight on the lesion region.")
print("GAP would dilute the lesion signal across the entire patch.")

---
## 7. Building Block 4: Learned Positional Encoding

### The problem
CNNs are **translation-equivariant**: the same feature activates the same way
regardless of where it appears in the patch. But spatial context matters in
pathology: invasive tumour fronts appear at patch edges, gland lumina appear
centrally surrounded by epithelium, stroma appears in characteristic spatial
relationships to tumour.

### The solution: learnable position bias
A trainable `(1, H, W, C)` weight matrix is added element-wise to the fused
feature map. It is initialised to near-zero and learns, from training data,
which spatial positions tend to be associated with each class. Unlike sinusoidal
encodings (which encode absolute position), this adapts to the actual spatial
statistics of your dataset.

In [ ]:
class LearnedPositionalEncoding(tfkl.Layer):
    """
    Trainable 2D spatial position bias.

    Adds a (1, H, W, C) learnable weight matrix to the feature map.
    Initialised with small random values (stddev=0.02, following ViT).
    After training, channels will have non-zero spatial patterns reflecting
    which positions correlate with the training labels.
    """

    def build(self, input_shape):
        _, H, W, C = input_shape
        self.pos_embed = self.add_weight(
            name        = "pos_embed",
            shape       = (1, H, W, C),
            initializer = keras.initializers.TruncatedNormal(stddev=0.02),
            trainable   = True,
        )

    def call(self, inputs):
        return inputs + self.pos_embed   # broadcast over batch dimension


# Visualise: what the positional encoding looks like before and after training
# Before training it is near-zero noise; after training it encodes spatial priors
pe_layer = LearnedPositionalEncoding()
dummy_in = keras.Input((8, 8, 16))
pe_out   = pe_layer(dummy_in)
pe_model = keras.Model(dummy_in, pe_out)
pe_model.predict(np.zeros((1, 8, 8, 16)), verbose=0)  # trigger build

# Visualise the initial (random) positional embedding for 4 channels
pe_weights = pe_layer.pos_embed.numpy()[0]  # (H, W, C)

fig, axes = plt.subplots(1, 4, figsize=(13, 3))
fig.suptitle("Learned Positional Encoding — 4 of 16 channels (random initialisation)\n"
             "After training these encode spatial priors (e.g. edge vs centre)",
             fontsize=11, fontweight="bold")

for i, ax in enumerate(axes):
    im = ax.imshow(pe_weights[:, :, i], cmap="RdBu_r")
    ax.set_title(f"Channel {i}")
    ax.axis("off")
    plt.colorbar(im, ax=ax, fraction=0.046)

plt.tight_layout()
plt.show()
print("Initial: near-zero noise. After training: structured spatial patterns.")
print("These patterns encode WHERE in the patch each feature tends to appear.")

---
## 8. Handling Class Imbalance

Our dataset is 25% cancerous / 75% benign — a realistic but challenging imbalance.

### Why standard cross-entropy fails
With 75% benign, a model that always predicts "benign" achieves 75% accuracy
while having 0% recall on the class that matters clinically.

### Three complementary strategies

**Strategy 1: Focal Loss** (loss level)  
Adds a modulating factor `(1 - p)^γ` to BCE that down-weights easy, well-classified
examples (the abundant benign patches) and focuses training on hard ones.

**Strategy 2: class_weight** (sample level)  
Keras multiplies each sample's loss by its class weight, so the model sees
the minority class as more important during gradient updates.

**Strategy 3: Oversampling** (dataset level)  
Repeat minority-class samples so both classes appear equally often per epoch.

In [ ]:
class FocalLoss(keras.losses.Loss):
    """
    Binary Focal Loss (Lin et al., 2017).

    Standard BCE:  L = -[y·log(p) + (1-y)·log(1-p)]
    Focal Loss:    L = -[y·(1-p)^γ·log(p) + (1-y)·p^γ·log(1-p)]

    (1-p)^γ → 0 for easy examples (p near 1 for positive, p near 0 for negative)
    When γ=0, Focal Loss equals standard BCE.
    """

    def __init__(self, gamma=2.0, alpha=0.25, **kwargs):
        super().__init__(**kwargs)
        self.gamma = gamma
        self.alpha = alpha

    def call(self, y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)
        ce_pos = -tf.math.log(y_pred)
        ce_neg = -tf.math.log(1.0 - y_pred)
        fl_pos = self.alpha       * tf.pow(1.0 - y_pred, self.gamma) * ce_pos
        fl_neg = (1 - self.alpha) * tf.pow(y_pred,       self.gamma) * ce_neg
        return tf.reduce_mean(y_true * fl_pos + (1 - y_true) * fl_neg)

    def get_config(self):
        return {**super().get_config(), "gamma": self.gamma, "alpha": self.alpha}


# Visualise: loss landscape for BCE vs Focal Loss
p     = np.linspace(0.01, 0.99, 200)   # predicted probability
bce   = -np.log(p)                     # BCE for positive class
fl2   = (1 - p) ** 2 * (-np.log(p))   # Focal (γ=2) for positive class
fl5   = (1 - p) ** 5 * (-np.log(p))   # Focal (γ=5) for positive class

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle("Focal Loss — down-weighting easy examples",
             fontsize=12, fontweight="bold")

axes[0].plot(p, bce,  label="BCE (γ=0)",      lw=2, color="black")
axes[0].plot(p, fl2,  label="Focal (γ=2)",    lw=2, color="darkorange")
axes[0].plot(p, fl5,  label="Focal (γ=5)",    lw=2, color="red")
axes[0].axvspan(0.7, 1.0, alpha=0.1, color="green",
                label="Easy examples (p>0.7)")
axes[0].set_xlabel("Predicted probability p (for positive class)")
axes[0].set_ylabel("Loss")
axes[0].set_title("Loss for positive-class examples")
axes[0].legend(); axes[0].set_ylim(0, 5)

# Show training dynamics: proportion of loss from each class with/without focal
# Dataset: 75% benign (easy), 25% cancerous (hard to learn)
frac_benign   = 0.75
p_benign_pred = 0.85    # model correctly predicts most benign as benign
p_cancer_pred = 0.45    # model is less confident on cancerous

bce_benign  = -np.log(1 - p_benign_pred)   # benign BCE (true label=0)
bce_cancer  = -np.log(p_cancer_pred)        # cancer BCE (true label=1)
fl_benign   = p_benign_pred**2 * bce_benign
fl_cancer   = (1-p_cancer_pred)**2 * bce_cancer

total_bce = frac_benign * bce_benign + (1-frac_benign) * bce_cancer
total_fl  = frac_benign * fl_benign  + (1-frac_benign) * fl_cancer

labels  = ["Benign", "Cancerous"]
bce_frac = [frac_benign * bce_benign / total_bce,
            (1-frac_benign) * bce_cancer / total_bce]
fl_frac  = [frac_benign * fl_benign  / total_fl,
            (1-frac_benign) * fl_cancer  / total_fl]

x = np.arange(2); w = 0.3
axes[1].bar(x - w/2, bce_frac, w, label="BCE",        color=["steelblue","tomato"])
axes[1].bar(x + w/2, fl_frac,  w, label="Focal (γ=2)",
            color=["steelblue","tomato"], alpha=0.5, hatch="//")
axes[1].set_xticks(x); axes[1].set_xticklabels(labels)
axes[1].set_ylabel("Fraction of total loss")
axes[1].set_title("Loss contribution per class\n(Focal reduces benign dominance)")
axes[1].legend()

plt.tight_layout()
plt.show()

# Compute class weights
n_total = len(y)
n_pos   = int(np.sum(y == 1))
n_neg   = int(np.sum(y == 0))
w_pos   = n_total / (2 * n_pos)
w_neg   = n_total / (2 * n_neg)
class_weights = {0: w_neg, 1: w_pos}
alpha_focal   = 1.0 - n_pos / n_total

print(f"Class weights — benign: {w_neg:.3f}  cancerous: {w_pos:.3f}")
print(f"Focal Loss alpha (auto): {alpha_focal:.3f}")
print(f"With γ=2, easy examples (p>0.7) contribute ~{(0.3**2):.2f}× less loss.")

---
## 9. Assembling and Training PathNet

Now we put all four blocks together into the full PathNet architecture and
train it on our simulated dataset using the two-stage protocol.

In [ ]:
def build_pathnet(input_shape=(64, 64, 3), classes=1,
                  fpn_channels=32, dropout_rate=0.4):
    """
    Lightweight PathNet for the tutorial (no pretrained backbone).

    In production, replace the three Conv encoder stages with EfficientNetB0
    skip connections (block3b_add, block5c_add, top_activation).
    Here we use a simple 3-stage Conv encoder that trains from scratch on
    our toy dataset.

    Architecture:
      Input
        → Stage 1: Conv 3×3 → MaxPool  (fine features)
        → Stage 2: Conv 3×3 → MaxPool  (mid features)
        → Stage 3: Conv 3×3            (coarse features)
        → Multi-Scale Fusion Block (per stage)
        → FPN: Concat + Conv 1×1
        → Learned Positional Encoding
        → SE Block
        → Spatial Attention Pooling
        → Dense head → Sigmoid
    """
    inputs = keras.Input(shape=input_shape, name="patch_input")

    # ── Encoder: 3 stages at different resolutions ────────────────────────────
    x = tfkl.Conv2D(32, 3, padding="same", activation="relu")(inputs)
    x = tfkl.BatchNormalization()(x)
    fine = tfkl.MaxPooling2D(2)(x)           # H/2, W/2 — nuclei-level

    x   = tfkl.Conv2D(64, 3, padding="same", activation="relu")(fine)
    x   = tfkl.BatchNormalization()(x)
    mid = tfkl.MaxPooling2D(2)(x)            # H/4, W/4 — cluster-level

    x      = tfkl.Conv2D(128, 3, padding="same", activation="relu")(mid)
    coarse = tfkl.BatchNormalization()(x)    # H/4, W/4 — tissue-level

    # ── Multi-Scale Fusion per scale ──────────────────────────────────────────
    fine_f   = MultiScaleFusionBlock(fpn_channels, name="msf_fine")(fine)
    mid_f    = MultiScaleFusionBlock(fpn_channels, name="msf_mid")(mid)
    coarse_f = MultiScaleFusionBlock(fpn_channels, name="msf_coarse")(coarse)

    # ── FPN: resize to coarse resolution, concat, project ─────────────────────
    target_h = tf.shape(coarse_f)[1]
    target_w = tf.shape(coarse_f)[2]
    fine_r   = tf.image.resize(fine_f,   [target_h, target_w])
    mid_r    = tf.image.resize(mid_f,    [target_h, target_w])

    fused = tfkl.Concatenate(axis=-1, name="fpn_concat")([fine_r, mid_r, coarse_f])
    fused = tfkl.Conv2D(fpn_channels, 1, padding="same",
                        use_bias=False, name="fpn_proj")(fused)
    fused = tfkl.BatchNormalization(name="fpn_bn")(fused)
    fused = tfkl.Activation("relu", name="fpn_relu")(fused)

    # ── Positional Encoding + SE + Spatial Attention Pooling ──────────────────
    fused  = LearnedPositionalEncoding(name="pos_enc")(fused)
    fused  = SqueezeExcitationBlock(ratio=4, name="fpn_se")(fused)
    pooled = SpatialAttentionPooling(name="spatial_attn_pool")(fused)

    # ── Classification head ───────────────────────────────────────────────────
    x = tfkl.Dense(64, activation="relu", name="head_fc1")(pooled)
    x = tfkl.Dropout(dropout_rate, name="head_drop")(x)
    x = tfkl.Dense(32, activation="relu", name="head_fc2")(x)
    out = tfkl.Dense(classes, activation="sigmoid", name="predictions")(x)

    return keras.Model(inputs, out, name="PathNet_tutorial")


model = build_pathnet(input_shape=(PATCH_SIZE, PATCH_SIZE, 3),
                      fpn_channels=32)
model.summary(line_length=80)
print(f"\nTotal parameters: {model.count_params():,}")

In [ ]:
# Train / val / test split — stratified to preserve class ratio
X_tv, X_test, y_tv, y_test = train_test_split(
    X_norm, y, test_size=0.15, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(
    X_tv, y_tv, test_size=0.2, random_state=42, stratify=y_tv)

print(f"Train : {len(X_train)} patches  "
      f"({int(np.sum(y_train==1))} cancerous / {int(np.sum(y_train==0))} benign)")
print(f"Val   : {len(X_val)} patches")
print(f"Test  : {len(X_test)} patches")

# Compile with Focal Loss + AUC metric
model.compile(
    optimizer = keras.optimizers.Adam(1e-3),
    loss      = FocalLoss(gamma=2.0, alpha=float(alpha_focal)),
    metrics   = [
        keras.metrics.AUC(name="auc"),
        keras.metrics.BinaryAccuracy(name="accuracy"),
        keras.metrics.Recall(name="recall"),
    ],
)

callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_auc", patience=10,
        mode="max", restore_best_weights=True, verbose=1,
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_auc", factor=0.5,
        patience=5, mode="max", verbose=1,
    ),
]

print("\nTraining PathNet (early stopping on val_auc)...")
history = model.fit(
    X_train, y_train,
    validation_data = (X_val, y_val),
    epochs          = 60,
    batch_size      = 32,
    class_weight    = class_weights,    # sample-level imbalance correction
    callbacks       = callbacks,
    verbose         = 1,
)

In [ ]:
# Training curves
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle("PathNet Training Curves", fontsize=13, fontweight="bold")

for ax, metric, title in zip(
    axes,
    ["loss", "auc", "recall"],
    ["Focal Loss", "AUC-ROC", "Recall (cancerous)"],
):
    ax.plot(history.history[metric],         label="train", lw=2)
    ax.plot(history.history[f"val_{metric}"], label="val",   lw=2, linestyle="--")
    ax.set_title(title); ax.set_xlabel("Epoch")
    ax.legend(); ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

---
## 10. Evaluation

### Why 0.5 is the wrong threshold
After Focal Loss + class_weight training on imbalanced data, the model's output
distribution shifts — predictions are no longer calibrated around 0.5. We use
the precision-recall curve on the **validation set** to find the threshold that
maximises F1, then apply it to the held-out **test set**.

In [ ]:
# Find optimal threshold on validation set
y_val_prob = model.predict(X_val, verbose=0).flatten()
prec, rec, thresholds = precision_recall_curve(y_val, y_val_prob)
f1_scores  = (2 * prec * rec) / (prec + rec + 1e-8)
best_idx   = int(np.argmax(f1_scores[:-1]))
opt_thresh = float(thresholds[best_idx])

print(f"Optimal threshold (val set, max F1): {opt_thresh:.3f}")
print(f"F1 at optimal threshold: {f1_scores[best_idx]:.4f}")
print(f"\n(Default 0.5 F1: {f1_score(y_val, (y_val_prob>=0.5).astype(int)):.4f})")

# Precision-recall curve with threshold shown
fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(rec, prec, lw=2, color="darkorange", label="Precision-Recall")
ax.scatter(rec[best_idx], prec[best_idx], s=120, color="red", zorder=5,
           label=f"Optimal threshold = {opt_thresh:.2f}")
ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
ax.set_title("Precision-Recall Curve (validation set)", fontweight="bold")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Full test-set evaluation with optimal threshold
y_test_prob = model.predict(X_test, verbose=0).flatten()
y_test_pred = (y_test_prob >= opt_thresh).astype(int)

auc = roc_auc_score(y_test, y_test_prob)
f1  = f1_score(y_test, y_test_pred)

print(f"Test AUC-ROC : {auc:.4f}")
print(f"Test F1      : {f1:.4f}  (threshold = {opt_thresh:.3f})")
print()
print(classification_report(y_test, y_test_pred,
                             target_names=["benign", "cancerous"]))

# ROC curve + confusion matrix
fpr, tpr, thr_roc = roc_curve(y_test, y_test_prob)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle("Test Set Evaluation", fontsize=13, fontweight="bold")

axes[0].plot(fpr, tpr, color="darkorange", lw=2,
             label=f"AUC = {auc:.3f}")
axes[0].plot([0, 1], [0, 1], "--", color="navy", lw=1, label="Random")
op_idx = int(np.argmin(np.abs(thr_roc - opt_thresh)))
axes[0].scatter(fpr[op_idx], tpr[op_idx], s=100, color="red", zorder=5,
                label=f"Threshold = {opt_thresh:.2f}")
axes[0].set_xlabel("FPR"); axes[0].set_ylabel("TPR")
axes[0].set_title("ROC Curve"); axes[0].legend()

ConfusionMatrixDisplay(
    confusion_matrix(y_test, y_test_pred),
    display_labels=["benign", "cancerous"],
).plot(ax=axes[1], colorbar=False)
axes[1].set_title(f"Confusion Matrix (threshold={opt_thresh:.2f})")

plt.tight_layout()
plt.show()

In [ ]:
# Visualise spatial attention maps on test patches
# Build a sub-model that outputs the attention map from SpatialAttentionPooling
attn_extractor = keras.Model(
    inputs  = model.input,
    outputs = model.get_layer("spatial_attn_pool").attn_conv.output,
)

# Pick 3 cancerous and 3 benign test patches
can_idx = np.where(y_test == 1)[0][:3]
ben_idx = np.where(y_test == 0)[0][:3]
show_idx = list(can_idx) + list(ben_idx)
titles   = ["Cancerous"] * 3 + ["Benign"] * 3

fig, axes = plt.subplots(3, len(show_idx), figsize=(15, 7))
fig.suptitle("Spatial Attention Maps — where is PathNet looking?",
             fontsize=13, fontweight="bold")

for col, (idx, title) in enumerate(zip(show_idx, titles)):
    patch = X_test[idx]
    prob  = float(y_test_prob[np.where(
        np.all(X_test == patch.reshape(1, *patch.shape), axis=(1,2,3))
    )[0][0]])

    # Extract attention map
    attn  = attn_extractor.predict(
        patch[np.newaxis], verbose=0)[0, :, :, 0]
    attn  = cv2.resize(attn, (PATCH_SIZE, PATCH_SIZE),
                       interpolation=cv2.INTER_LINEAR)
    attn  = (attn - attn.min()) / (attn.max() - attn.min() + 1e-8)

    # Overlay
    heatmap = cm.jet(attn)[:, :, :3].astype(np.float32)
    overlay = np.clip(0.55 * patch + 0.45 * heatmap, 0, 1)

    color = "red" if title == "Cancerous" else "green"

    axes[0, col].imshow(patch)
    axes[0, col].set_title(f"{title}\np={prob:.2f}", color=color, fontsize=9)
    axes[0, col].axis("off")

    axes[1, col].imshow(attn, cmap="jet")
    axes[1, col].set_title("Attention map", fontsize=9)
    axes[1, col].axis("off")

    axes[2, col].imshow(overlay)
    axes[2, col].set_title("Overlay", fontsize=9)
    axes[2, col].axis("off")

row_labels = ["Original patch", "Attention map", "Overlay"]
for row, label in enumerate(row_labels):
    axes[row, 0].set_ylabel(label, fontsize=10, fontweight="bold")

plt.tight_layout()
plt.show()
print("High-attention regions (red/yellow) show where PathNet focused its classification.")
print("For cancerous patches, these should correspond to dense, irregular nuclear regions.")

---
## Summary

Here is what every architectural choice in PathNet was designed to solve:

| Component | Biological problem | Engineering solution |
|---|---|---|
| **Macenko normalisation** | Scanner colour variation causes domain shift | SVD stain decomposition; reconstruct with reference vectors |
| **Multi-Scale Fusion (3×3/5×5/7×7)** | Cancer is visible at nucleus AND tissue scale simultaneously | Parallel branches + concat + 1×1 projection |
| **Concat (not sum) fusion** | Scales encode complementary, not identical information | Let 1×1 Conv learn the combination |
| **Squeeze-and-Excitation** | Staining artefacts create noisy feature channels | Per-channel sigmoid gate conditioned on global context |
| **Learned Positional Encoding** | CNN has no spatial memory; tissue patterns are position-sensitive | Trainable (H,W,C) bias matrix |
| **Spatial Attention Pooling** | GAP dilutes rare lesion signal with majority background | Learned α(h,w) weight map |
| **Focal Loss + class_weight** | 75% benign → model predicts benign for everything | Down-weight easy examples; up-weight minority class |
| **Optimal threshold** | 0.5 is miscalibrated after imbalance correction | Sweep PR curve on val set; maximise F1 |

---

## Adapting to real data

To use this tutorial on real WSI data:

1. Replace `generate_dataset()` with `load_patch_dataset()` from `data.py`
2. Replace the 3-stage Conv encoder with `EfficientNetB0` skip connections:
   - `block3b_add` → fine (28×28×40)
   - `block5c_add` → mid (14×14×112)
   - `top_activation` → coarse (7×7×1280)
3. Use the full `train.py` two-stage fine-tuning protocol
4. See the companion `PathNet/` repository for the production codebase

---
*This notebook uses only simulated data. No patient data or proprietary slides are included.*